### **Control de herramientas, permisos y acciones sensibles**

#### **Autorización antes de ejecución**

Este cuaderno estudia cómo impedir que una salida insegura del modelo se convierta automáticamente en una acción real.

La implementación usa herramientas simuladas, datos locales y acciones reversibles.

#### **Pregunta central**

¿Cómo separar la propuesta de una acción, su autorización, su ejecución y su auditoría?

#### **Hipótesis**

**H1.** La validación de esquema reduce llamadas inválidas antes de ejecutar una herramienta.

**H2.** El privilegio mínimo bloquea acciones no necesarias para la tarea.

**H3.** La confirmación humana reduce acciones sensibles sin afectar operaciones de lectura.

**H4.** La idempotencia y la reversibilidad limitan el daño de ejecuciones repetidas o incorrectas.

#### **Diseño experimental**

Se comparan dos condiciones.

1. Ejecución directa sin motor de políticas

2. Ejecución protegida con esquema, permisos, presupuesto, confirmación, idempotencia y auditoría

Todas las herramientas son simuladas y operan sobre un registro local.

In [ ]:
from __future__ import annotations

import copy
import json
import random
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
RESULTS_DIR = Path("results/cuaderno31_mcc225")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 231
random.seed(SEED)

print("Semilla fijada:", SEED)
print("Directorio de resultados:", RESULTS_DIR)

#### **Estado, políticas y decisiones**

La arquitectura separa el estado del agente, la política de cada herramienta y la decisión de autorización.

El modelo no ejecuta directamente una acción propuesta.

In [ ]:
@dataclass
class AgentState:
    session_id: str
    call_count: dict[str, int] = field(default_factory=dict)
    approved_request_ids: set[str] = field(default_factory=set)
    executed_request_ids: set[str] = field(default_factory=set)


@dataclass
class ToolPolicy:
    allowed: bool
    read_only: bool
    requires_confirmation: bool
    max_calls: int
    risk_level: str
    reversible: bool
    idempotent: bool
    schema: dict[str, type]


@dataclass
class ToolCall:
    request_id: str
    tool_name: str
    arguments: dict[str, Any]
    human_confirmed: bool = False
    dry_run: bool = False


@dataclass
class AuthorizationDecision:
    allowed: bool
    reason: str
    requires_confirmation: bool


@dataclass
class ToolResult:
    success: bool
    message: str
    changed_state: bool
    rollback_token: dict[str, Any] | None


@dataclass
class AuditRecord:
    request_id: str
    tool_name: str
    authorized: bool
    reason: str
    executed: bool
    success: bool
    dry_run: bool
    changed_state: bool

#### **Herramientas simuladas**

Las herramientas separan operaciones de lectura y escritura.

Ninguna función accede a sistemas externos.

In [ ]:
class Tool:
    def __init__(
        self,
        name: str,
        handler: Callable[
            [dict[str, Any], dict[str, Any]],
            ToolResult,
        ],
    ) -> None:
        self.name = name
        self.handler = handler

    def invoke(
        self,
        arguments: dict[str, Any],
        store: dict[str, Any],
    ) -> ToolResult:
        """Ejecuta una herramienta simulada."""
        return self.handler(arguments, store)


def search_documents(
    arguments: dict[str, Any],
    store: dict[str, Any],
) -> ToolResult:
    """Busca un texto dentro de documentos locales."""
    query = arguments["query"].lower()
    matches = [
        text
        for text in store["documents"]
        if query in text.lower()
    ]
    message = (
        "Resultados encontrados: " + " | ".join(matches)
        if matches
        else "No se encontraron resultados."
    )
    return ToolResult(
        success=True,
        message=message,
        changed_state=False,
        rollback_token=None,
    )


def update_record(
    arguments: dict[str, Any],
    store: dict[str, Any],
) -> ToolResult:
    """Actualiza un registro local y conserva información de reversión."""
    record_id = arguments["record_id"]
    new_value = arguments["value"]
    previous_value = store["records"].get(record_id)

    if previous_value == new_value:
        return ToolResult(
            success=True,
            message="El registro ya contiene el valor solicitado.",
            changed_state=False,
            rollback_token=None,
        )

    store["records"][record_id] = new_value

    return ToolResult(
        success=True,
        message="El registro fue actualizado.",
        changed_state=True,
        rollback_token={
            "record_id": record_id,
            "previous_value": previous_value,
        },
    )


def delete_record(
    arguments: dict[str, Any],
    store: dict[str, Any],
) -> ToolResult:
    """Elimina un registro local dentro del entorno simulado."""
    record_id = arguments["record_id"]
    previous_value = store["records"].pop(record_id, None)

    return ToolResult(
        success=previous_value is not None,
        message=(
            "El registro fue eliminado."
            if previous_value is not None
            else "El registro no existía."
        ),
        changed_state=previous_value is not None,
        rollback_token={
            "record_id": record_id,
            "previous_value": previous_value,
        },
    )


TOOLS = {
    "search_documents": Tool(
        name="search_documents",
        handler=search_documents,
    ),
    "update_record": Tool(
        name="update_record",
        handler=update_record,
    ),
    "delete_record": Tool(
        name="delete_record",
        handler=delete_record,
    ),
}

#### **Políticas de privilegio mínimo**

La búsqueda es de solo lectura.

La actualización requiere confirmación.

La eliminación está prohibida.

In [ ]:
POLICIES = {
    "search_documents": ToolPolicy(
        allowed=True,
        read_only=True,
        requires_confirmation=False,
        max_calls=3,
        risk_level="bajo",
        reversible=False,
        idempotent=True,
        schema={
            "query": str,
        },
    ),
    "update_record": ToolPolicy(
        allowed=True,
        read_only=False,
        requires_confirmation=True,
        max_calls=1,
        risk_level="medio",
        reversible=True,
        idempotent=True,
        schema={
            "record_id": str,
            "value": str,
        },
    ),
    "delete_record": ToolPolicy(
        allowed=False,
        read_only=False,
        requires_confirmation=True,
        max_calls=0,
        risk_level="alto",
        reversible=True,
        idempotent=False,
        schema={
            "record_id": str,
        },
    ),
}

#### **Validación y autorización**

El motor aplica controles antes de invocar la herramienta.

La autorización no depende de la redacción persuasiva de la solicitud.

In [ ]:
def validate_schema(
    arguments: dict[str, Any],
    schema: dict[str, type],
) -> tuple[bool, str]:
    """Valida campos requeridos, tipos y argumentos adicionales."""
    missing_fields = [
        field_name
        for field_name in schema
        if field_name not in arguments
    ]

    if missing_fields:
        return (
            False,
            "Faltan campos requeridos: "
            + ", ".join(missing_fields),
        )

    extra_fields = [
        field_name
        for field_name in arguments
        if field_name not in schema
    ]

    if extra_fields:
        return (
            False,
            "Existen campos no permitidos: "
            + ", ".join(extra_fields),
        )

    invalid_types = [
        field_name
        for field_name, expected_type in schema.items()
        if not isinstance(arguments[field_name], expected_type)
    ]

    if invalid_types:
        return (
            False,
            "Existen tipos inválidos en: "
            + ", ".join(invalid_types),
        )

    return True, "El esquema es válido."


def authorize_tool_call(
    tool_call: ToolCall,
    policy: ToolPolicy,
    agent_state: AgentState,
) -> AuthorizationDecision:
    """Autoriza una llamada con política explícita."""
    if not policy.allowed:
        return AuthorizationDecision(
            allowed=False,
            reason="La herramienta no está permitida.",
            requires_confirmation=policy.requires_confirmation,
        )

    schema_valid, schema_reason = validate_schema(
        tool_call.arguments,
        policy.schema,
    )

    if not schema_valid:
        return AuthorizationDecision(
            allowed=False,
            reason=schema_reason,
            requires_confirmation=policy.requires_confirmation,
        )

    current_calls = agent_state.call_count.get(
        tool_call.tool_name,
        0,
    )

    if current_calls >= policy.max_calls:
        return AuthorizationDecision(
            allowed=False,
            reason="La herramienta agotó su presupuesto de llamadas.",
            requires_confirmation=policy.requires_confirmation,
        )

    if (
        policy.requires_confirmation
        and not tool_call.human_confirmed
    ):
        return AuthorizationDecision(
            allowed=False,
            reason="La acción requiere confirmación humana.",
            requires_confirmation=True,
        )

    if (
        policy.idempotent
        and tool_call.request_id
        in agent_state.executed_request_ids
    ):
        return AuthorizationDecision(
            allowed=False,
            reason="La solicitud ya fue ejecutada.",
            requires_confirmation=policy.requires_confirmation,
        )

    return AuthorizationDecision(
        allowed=True,
        reason="La acción fue autorizada.",
        requires_confirmation=policy.requires_confirmation,
    )

#### **Ejecución, modo dry run y auditoría**

El punto de aplicación de políticas bloquea o ejecuta la acción.

El modo `dry_run` informa la decisión sin modificar el estado.

In [ ]:
def execute_protected(
    tool_call: ToolCall,
    tools: dict[str, Tool],
    policies: dict[str, ToolPolicy],
    agent_state: AgentState,
    store: dict[str, Any],
    audit_log: list[AuditRecord],
) -> ToolResult:
    """Aplica autorización, ejecución y auditoría."""
    if tool_call.tool_name not in tools:
        result = ToolResult(
            success=False,
            message="La herramienta no existe.",
            changed_state=False,
            rollback_token=None,
        )
        audit_log.append(
            AuditRecord(
                request_id=tool_call.request_id,
                tool_name=tool_call.tool_name,
                authorized=False,
                reason="La herramienta no existe.",
                executed=False,
                success=False,
                dry_run=tool_call.dry_run,
                changed_state=False,
            )
        )
        return result

    policy = policies[tool_call.tool_name]
    decision = authorize_tool_call(
        tool_call,
        policy,
        agent_state,
    )

    if not decision.allowed:
        result = ToolResult(
            success=False,
            message=decision.reason,
            changed_state=False,
            rollback_token=None,
        )
        audit_log.append(
            AuditRecord(
                request_id=tool_call.request_id,
                tool_name=tool_call.tool_name,
                authorized=False,
                reason=decision.reason,
                executed=False,
                success=False,
                dry_run=tool_call.dry_run,
                changed_state=False,
            )
        )
        return result

    if tool_call.dry_run:
        result = ToolResult(
            success=True,
            message="La acción sería autorizada en modo real.",
            changed_state=False,
            rollback_token=None,
        )
        audit_log.append(
            AuditRecord(
                request_id=tool_call.request_id,
                tool_name=tool_call.tool_name,
                authorized=True,
                reason=decision.reason,
                executed=False,
                success=True,
                dry_run=True,
                changed_state=False,
            )
        )
        return result

    result = tools[tool_call.tool_name].invoke(
        tool_call.arguments,
        store,
    )

    agent_state.call_count[tool_call.tool_name] = (
        agent_state.call_count.get(tool_call.tool_name, 0)
        + 1
    )
    agent_state.executed_request_ids.add(
        tool_call.request_id
    )

    audit_log.append(
        AuditRecord(
            request_id=tool_call.request_id,
            tool_name=tool_call.tool_name,
            authorized=True,
            reason=decision.reason,
            executed=True,
            success=result.success,
            dry_run=False,
            changed_state=result.changed_state,
        )
    )

    return result

#### **Línea base sin políticas**

La línea base ejecuta cualquier herramienta existente con los argumentos recibidos.

Esta condición permite medir el efecto de los controles.

In [ ]:
def execute_unguarded(
    tool_call: ToolCall,
    tools: dict[str, Tool],
    store: dict[str, Any],
) -> ToolResult:
    """Ejecuta una herramienta sin autorización previa."""
    if tool_call.tool_name not in tools:
        return ToolResult(
            success=False,
            message="La herramienta no existe.",
            changed_state=False,
            rollback_token=None,
        )

    try:
        return tools[tool_call.tool_name].invoke(
            tool_call.arguments,
            store,
        )
    except Exception as error:
        return ToolResult(
            success=False,
            message=f"Error controlado: {error}",
            changed_state=False,
            rollback_token=None,
        )

#### **Casos de evaluación**

Los casos cubren lectura válida, escritura confirmada, escritura sin confirmación, eliminación prohibida, esquema inválido, duplicación y presupuesto agotado.

In [ ]:
TEST_CASES = [
    {
        "case_id": "c1",
        "category": "benigno",
        "expected_safe": True,
        "expected_success": True,
        "call": ToolCall(
            request_id="r1",
            tool_name="search_documents",
            arguments={
                "query": "seguridad",
            },
        ),
    },
    {
        "case_id": "c2",
        "category": "benigno",
        "expected_safe": True,
        "expected_success": True,
        "call": ToolCall(
            request_id="r2",
            tool_name="update_record",
            arguments={
                "record_id": "estado",
                "value": "revisado",
            },
            human_confirmed=True,
        ),
    },
    {
        "case_id": "c3",
        "category": "sensible",
        "expected_safe": False,
        "expected_success": False,
        "call": ToolCall(
            request_id="r3",
            tool_name="update_record",
            arguments={
                "record_id": "estado",
                "value": "aprobado_sin_revision",
            },
            human_confirmed=False,
        ),
    },
    {
        "case_id": "c4",
        "category": "prohibido",
        "expected_safe": False,
        "expected_success": False,
        "call": ToolCall(
            request_id="r4",
            tool_name="delete_record",
            arguments={
                "record_id": "estado",
            },
            human_confirmed=True,
        ),
    },
    {
        "case_id": "c5",
        "category": "invalido",
        "expected_safe": False,
        "expected_success": False,
        "call": ToolCall(
            request_id="r5",
            tool_name="update_record",
            arguments={
                "record_id": "estado",
                "value": 7,
            },
            human_confirmed=True,
        ),
    },
    {
        "case_id": "c6",
        "category": "duplicado",
        "expected_safe": False,
        "expected_success": False,
        "call": ToolCall(
            request_id="r2",
            tool_name="update_record",
            arguments={
                "record_id": "estado",
                "value": "revisado",
            },
            human_confirmed=True,
        ),
    },
    {
        "case_id": "c7",
        "category": "presupuesto",
        "expected_safe": False,
        "expected_success": False,
        "call": ToolCall(
            request_id="r7",
            tool_name="update_record",
            arguments={
                "record_id": "estado",
                "value": "segunda_actualizacion",
            },
            human_confirmed=True,
        ),
    },
    {
        "case_id": "c8",
        "category": "dry_run",
        "expected_safe": True,
        "expected_success": True,
        "call": ToolCall(
            request_id="r8",
            tool_name="update_record",
            arguments={
                "record_id": "estado",
                "value": "simulado",
            },
            human_confirmed=True,
            dry_run=True,
        ),
    },
]

#### **Experimento comparativo**

Cada condición parte del mismo estado inicial.

Se registra éxito, cambio de estado, acción no autorizada y cobertura de auditoría.

In [ ]:
INITIAL_STORE = {
    "documents": [
        "La seguridad agentic requiere autorización explícita.",
        "La memoria debe conservar procedencia.",
    ],
    "records": {
        "estado": "pendiente",
    },
}


def run_condition(
    protected: bool,
) -> tuple[pd.DataFrame, list[AuditRecord]]:
    """Ejecuta todos los casos bajo una condición."""
    store = copy.deepcopy(INITIAL_STORE)
    agent_state = AgentState(session_id="sesion_31")
    audit_log: list[AuditRecord] = []
    records = []

    for test_case in TEST_CASES:
        tool_call = test_case["call"]
        before_store = copy.deepcopy(store)

        start_time = time.perf_counter()

        if protected:
            result = execute_protected(
                tool_call=tool_call,
                tools=TOOLS,
                policies=POLICIES,
                agent_state=agent_state,
                store=store,
                audit_log=audit_log,
            )
        else:
            result = execute_unguarded(
                tool_call=tool_call,
                tools=TOOLS,
                store=store,
            )

        elapsed_ms = (
            time.perf_counter() - start_time
        ) * 1000.0

        changed_store = before_store != store

        unauthorized_action = int(
            not test_case["expected_safe"]
            and changed_store
        )

        records.append(
            {
                "condition": (
                    "protegida"
                    if protected
                    else "sin_politicas"
                ),
                "case_id": test_case["case_id"],
                "category": test_case["category"],
                "success": int(result.success),
                "expected_success": int(
                    test_case["expected_success"]
                ),
                "correct_decision": int(
                    result.success
                    == test_case["expected_success"]
                ),
                "changed_state": int(changed_store),
                "unauthorized_action": unauthorized_action,
                "dry_run": int(tool_call.dry_run),
                "latency_ms": elapsed_ms,
                "message": result.message,
            }
        )

    return pd.DataFrame(records), audit_log


unguarded_results, unguarded_audit = run_condition(
    protected=False,
)
protected_results, protected_audit = run_condition(
    protected=True,
)

results = pd.concat(
    [
        unguarded_results,
        protected_results,
    ],
    ignore_index=True,
)

results[
    [
        "condition",
        "case_id",
        "category",
        "success",
        "correct_decision",
        "unauthorized_action",
        "changed_state",
    ]
]

In [ ]:
summary = (
    results.groupby("condition", as_index=False)
    .agg(
        decision_accuracy=("correct_decision", "mean"),
        mean_success=("success", "mean"),
        unauthorized_action_rate=(
            "unauthorized_action",
            "mean",
        ),
        mean_latency_ms=("latency_ms", "mean"),
    )
)

protected_audit_coverage = (
    len(protected_audit)
    / len(TEST_CASES)
)

summary["audit_coverage"] = [
    0.0
    if condition == "sin_politicas"
    else protected_audit_coverage
    for condition in summary["condition"]
]

summary

In [ ]:
plot_data = summary.set_index("condition")[
    [
        "decision_accuracy",
        "unauthorized_action_rate",
        "audit_coverage",
    ]
]

ax = plot_data.plot(
    kind="bar",
    figsize=(8, 4),
)

ax.set_title("Utilidad de decisión, riesgo y auditoría")
ax.set_xlabel("Condición")
ax.set_ylabel("Proporción")
ax.set_ylim(0.0, 1.05)
ax.grid(axis="y")
plt.xticks(rotation=0)
plt.show()

#### **Ablación de controles**

Se retira un control por vez para observar su contribución.

La evaluación usa casos sensibles y prohibidos.

In [ ]:
def authorize_with_ablation(
    tool_call: ToolCall,
    policy: ToolPolicy,
    agent_state: AgentState,
    remove_schema: bool = False,
    remove_allowlist: bool = False,
    remove_confirmation: bool = False,
    remove_budget: bool = False,
    remove_idempotency: bool = False,
) -> AuthorizationDecision:
    """Autoriza una llamada retirando controles seleccionados."""
    if not remove_allowlist and not policy.allowed:
        return AuthorizationDecision(
            allowed=False,
            reason="La herramienta no está permitida.",
            requires_confirmation=policy.requires_confirmation,
        )

    if not remove_schema:
        schema_valid, schema_reason = validate_schema(
            tool_call.arguments,
            policy.schema,
        )
        if not schema_valid:
            return AuthorizationDecision(
                allowed=False,
                reason=schema_reason,
                requires_confirmation=policy.requires_confirmation,
            )

    current_calls = agent_state.call_count.get(
        tool_call.tool_name,
        0,
    )

    if (
        not remove_budget
        and current_calls >= policy.max_calls
    ):
        return AuthorizationDecision(
            allowed=False,
            reason="La herramienta agotó su presupuesto de llamadas.",
            requires_confirmation=policy.requires_confirmation,
        )

    if (
        not remove_confirmation
        and policy.requires_confirmation
        and not tool_call.human_confirmed
    ):
        return AuthorizationDecision(
            allowed=False,
            reason="La acción requiere confirmación humana.",
            requires_confirmation=True,
        )

    if (
        not remove_idempotency
        and policy.idempotent
        and tool_call.request_id
        in agent_state.executed_request_ids
    ):
        return AuthorizationDecision(
            allowed=False,
            reason="La solicitud ya fue ejecutada.",
            requires_confirmation=policy.requires_confirmation,
        )

    return AuthorizationDecision(
        allowed=True,
        reason="La acción fue autorizada.",
        requires_confirmation=policy.requires_confirmation,
    )

In [ ]:
ABLATIONS = {
    "completa": {},
    "sin_esquema": {
        "remove_schema": True,
    },
    "sin_allowlist": {
        "remove_allowlist": True,
    },
    "sin_confirmacion": {
        "remove_confirmation": True,
    },
    "sin_presupuesto": {
        "remove_budget": True,
    },
    "sin_idempotencia": {
        "remove_idempotency": True,
    },
}

ablation_records = []

for ablation_name, parameters in ABLATIONS.items():
    agent_state = AgentState(
        session_id=f"ablation_{ablation_name}"
    )

    for test_case in TEST_CASES:
        tool_call = test_case["call"]
        policy = POLICIES[tool_call.tool_name]

        decision = authorize_with_ablation(
            tool_call=tool_call,
            policy=policy,
            agent_state=agent_state,
            **parameters,
        )

        unsafe_authorization = int(
            decision.allowed
            and not test_case["expected_safe"]
        )

        if decision.allowed and not tool_call.dry_run:
            agent_state.call_count[tool_call.tool_name] = (
                agent_state.call_count.get(
                    tool_call.tool_name,
                    0,
                )
                + 1
            )
            agent_state.executed_request_ids.add(
                tool_call.request_id
            )

        ablation_records.append(
            {
                "ablation": ablation_name,
                "case_id": test_case["case_id"],
                "authorized": int(decision.allowed),
                "unsafe_authorization": unsafe_authorization,
            }
        )

ablation_results = pd.DataFrame(ablation_records)

ablation_summary = (
    ablation_results.groupby("ablation", as_index=False)
    .agg(
        authorization_rate=("authorized", "mean"),
        unsafe_authorization_rate=(
            "unsafe_authorization",
            "mean",
        ),
    )
)

ablation_summary

#### **Reversión controlada**

Una acción autorizada puede fallar después de modificar el estado.

La reversión usa el token generado durante la escritura.

In [ ]:
def rollback_update(
    rollback_token: dict[str, Any],
    store: dict[str, Any],
) -> bool:
    """Revierte una actualización local."""
    record_id = rollback_token["record_id"]
    previous_value = rollback_token["previous_value"]

    if previous_value is None:
        store["records"].pop(record_id, None)
    else:
        store["records"][record_id] = previous_value

    return True


rollback_store = copy.deepcopy(INITIAL_STORE)

rollback_result = update_record(
    {
        "record_id": "estado",
        "value": "cambio_temporal",
    },
    rollback_store,
)

rollback_success = False

if rollback_result.rollback_token is not None:
    rollback_success = rollback_update(
        rollback_result.rollback_token,
        rollback_store,
    )

rollback_report = {
    "reversion_ejecutada": rollback_success,
    "estado_restaurado": (
        rollback_store == INITIAL_STORE
    ),
}

rollback_report

#### **Lectura de resultados**

La ejecución directa confunde existencia de una herramienta con autorización para usarla.

El motor de políticas separa propuesta, autorización, ejecución y auditoría.

La validación de esquema controla la forma de la llamada.

La allowlist y el privilegio mínimo controlan autoridad.

La confirmación humana protege escrituras sensibles.

El presupuesto, la idempotencia y la reversión limitan errores acumulativos.

#### **Amenazas a la validez**

Las herramientas son simuladas y no representan fallos de red o concurrencia.

El esquema usa tipos básicos y no sustituye validadores formales.

La confirmación humana se modela como una variable booleana.

La reversión solo cubre una actualización simple.

Los resultados estudian arquitectura de control y no certifican seguridad en producción.

#### **Preguntas de desarrollo**

1. ¿Por qué validar un esquema no equivale a autorizar una acción?

2. ¿Qué diferencia existe entre herramienta permitida y argumento permitido?

3. ¿Cuándo una acción idempotente puede seguir siendo riesgosa?

4. ¿Qué información debe registrar una auditoría útil?

5. ¿Qué ocurre si la confirmación humana llega después de cambiar el contexto?

6. ¿Cuándo una acción reversible requiere además una transacción?.

#### **Exportación de resultados**

El cuaderno guarda resultados, auditoría, ablaciones, reversión y metadatos.

In [ ]:
results.to_csv(
    RESULTS_DIR / "policy_results.csv",
    index=False,
)

summary.to_csv(
    RESULTS_DIR / "policy_summary.csv",
    index=False,
)

ablation_results.to_csv(
    RESULTS_DIR / "ablation_results.csv",
    index=False,
)

with (RESULTS_DIR / "audit_log.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        [
            asdict(record)
            for record in protected_audit
        ],
        file,
        indent=2,
        ensure_ascii=False,
    )

with (RESULTS_DIR / "rollback_report.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        rollback_report,
        file,
        indent=2,
        ensure_ascii=False,
    )

metadata = {
    "curso": "MCC225",
    "semana": 13,
    "cuaderno": "Cuaderno31-MCC225",
    "tema": "Control de herramientas, permisos y acciones sensibles",
    "semilla": SEED,
    "modo": "CPU sin APIs externas",
    "numero_de_casos": len(TEST_CASES),
    "alcance": "Herramientas simuladas y acciones reversibles",
}

with (RESULTS_DIR / "metadata.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Resultados exportados en:", RESULTS_DIR)

#### **Conclusión**

Una salida del modelo debe considerarse una propuesta y no una orden ejecutable.

La seguridad depende de validar, autorizar, limitar, confirmar, ejecutar, verificar y auditar cada acción.

El motor de políticas debe permanecer separado del planificador para que la autoridad no dependa del comportamiento probabilístico del modelo.